# Этап 5. Эксперименты с представлением треков

Вход: `data/features.csv`, `data/clap_embeddings.npy` (этап 3)

Выход: `data/experiment_results.json`, `data/similar_artists.csv`, графики

In [ ]:
%pip install -q "yt-dlp[default]" imageio-ffmpeg rapidfuzz librosa soundfile umap-learn transformers tqdm
%pip install -q deno

In [ ]:
import os, sys, json, time, re, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

IN_COLAB = 'google.colab' in sys.modules
USE_DRIVE = True

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/music_dataset')
else:
    BASE_DIR = Path('music_dataset').resolve()

AUDIO_DIR, DATA_DIR, FIG_DIR = BASE_DIR / 'audio', BASE_DIR / 'data', BASE_DIR / 'figures'
for d in (AUDIO_DIR, DATA_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

GENRE_ORDER = ['rock', 'pop', 'hip_hop', 'electronic', 'jazz', 'classical',
               'metal', 'rnb_soul', 'country', 'reggae', 'ambient', 'latin']
GENRE_RU = {'rock': 'рок', 'pop': 'поп', 'hip_hop': 'хип-хоп', 'electronic': 'электроника',
            'jazz': 'джаз', 'classical': 'классика', 'metal': 'метал', 'rnb_soul': 'R&B / соул',
            'country': 'кантри', 'reggae': 'регги', 'ambient': 'эмбиент', 'latin': 'латино'}

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

CAT = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
ACCENT = CAT[0]
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})

# жанров 12, а цветов 8, поэтому последние 4 жанра рисую треугольниками
def genre_style(g):
    i = GENRE_ORDER.index(g)
    return CAT[i % 8], ('o' if i < 8 else '^')

def savefig(name):
    plt.savefig(FIG_DIR / f'{name}.png', bbox_inches='tight')

def scatter_by_genre(ax, xy, genres, title):
    for g in GENRE_ORDER:
        m = np.asarray(genres) == g
        if m.any():
            c, mk = genre_style(g)
            ax.scatter(xy[m, 0], xy[m, 1], s=46, c=c, marker=mk, label=GENRE_RU[g],
                       edgecolors='white', linewidths=1)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

## 5. Похожесть треков

Похожесть считаю просто: косинусная близость и k ближайших соседей. Сравниваю несколько вариантов представления трека:
- эмбеддинг CLAP (512);
- семантические теги: настроение, контекст, атмосфера, инструменты, вокал (36 чисел);
- сигнальные признаки librosa (11 чисел);
- гибрид: все три блока, каждый нормирован и взят со своим весом;
- случайные векторы для сравнения.

Жанр от CLAP ни в один вариант не входит.

Данных о прослушиваниях у меня нет, поэтому качество оцениваю косвенно:
- **Genre P@k**: какая доля из k ближайших соседей того же жанра. Случайно получится примерно 0.07.
- **Era P@k**: то же самое для эпохи.
- Слушаю соседей для нескольких треков (ссылки в таблицах).
- Убираю группы признаков по очереди и смотрю, как меняется качество.

In [ ]:
import umap
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

features = pd.read_csv(DATA_DIR / 'features.csv')
E_all = np.load(DATA_DIR / 'clap_embeddings.npy')
ids = pd.read_csv(DATA_DIR / 'clap_embeddings_ids.csv').iloc[:, 0].tolist()
E = E_all[[ids.index(t) for t in features['track_id']]]
N = len(features)

MOODS = [c for c in features.columns if c.startswith('mood_') and not c.startswith('mood_top')]
CTXS = [c for c in features.columns if c.startswith('ctx_') and not c.startswith('ctx_top')]
ATMS = [c for c in features.columns if c.startswith('atm_') and not c.startswith('atm_top')]
INSTS = [c for c in features.columns if c.startswith('inst_')] + ['vocals_prob']
SEMANTIC = MOODS + CTXS + ATMS + INSTS
SIGNAL = ['tempo_bpm', 'rms_mean', 'rms_dynamic_range_db', 'spectral_centroid_hz', 'spectral_flatness',
          'onset_rate', 'pulse_clarity', 'percussive_ratio', 'zcr_mean', 'mode_major', 'key_strength']

def l2n(X):
    return X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-9, None)
def zs(X):
    X = np.nan_to_num(np.asarray(X, dtype=float), nan=np.nanmean(X))
    return (X - X.mean(0)) / np.clip(X.std(0), 1e-9, None)

B_clap, B_sem, B_sig = l2n(E), l2n(zs(features[SEMANTIC].values)), l2n(zs(features[SIGNAL].values))
W = {'clap': 1.0, 'sem': 0.7, 'sig': 0.4}

def hybrid(w=W):
    blocks = [B * w[k] for k, B in (('clap', B_clap), ('sem', B_sem), ('sig', B_sig)) if w.get(k, 0) > 0]
    return l2n(np.hstack(blocks))

rng = np.random.default_rng(SEED)
REPS = {
    'CLAP-эмбеддинг': B_clap,
    'Семантические теги': B_sem,
    'Сигнальные признаки': B_sig,
    'Гибрид': hybrid(),
    'Случайный': l2n(rng.normal(size=(N, 32))),
}
genres = features['genre'].to_numpy(dtype=str)
eras = features['era'].to_numpy(dtype=str)

In [ ]:
def neighbors(X, k):
    S = X @ X.T
    np.fill_diagonal(S, -np.inf)
    idx = np.argsort(-S, axis=1)[:, :k]
    return idx, np.take_along_axis(S, idx, axis=1)

def precision_at_k(X, labels, k=5):
    idx, _ = neighbors(X, k)
    return float((labels[idx] == labels[:, None]).mean())

rows = []
for name, X in REPS.items():
    rows.append({'представление': name, 'размерность': X.shape[1],
                 'Genre P@5': precision_at_k(X, genres, 5), 'Genre P@10': precision_at_k(X, genres, 10),
                 'Era P@5': precision_at_k(X, eras, 5)})
results = pd.DataFrame(rows).set_index('представление')
gc = pd.Series(genres).value_counts()
chance_genre = float((gc * (gc - 1)).sum() / (N * (N - 1)))
print(f'случайный уровень: {chance_genre:.3f}')
print(results.round(3))

fig, ax = plt.subplots(figsize=(9, 3.6))
r = results['Genre P@5'].sort_values()
ax.barh(r.index, r.values, color=[ACCENT if n != 'Случайный' else '#c9c8c3' for n in r.index], height=0.6)
ax.axvline(chance_genre, color='#52514e', ls='--', lw=1); ax.text(chance_genre, -0.7, ' случайно', fontsize=8, color='#52514e')
for y, v in enumerate(r.values):
    ax.text(v + 0.01, y, f'{v:.2f}', va='center', fontsize=9)
ax.set_xlim(0, 1); ax.set_xlabel('Genre P@5'); ax.set_title('Genre P@5 по представлениям')
plt.tight_layout(); savefig('10_similarity_precision'); plt.show()

### 5.1 UMAP и t-SNE
Цвет и маркер показывают жанр. Чем компактнее группы, тем лучше представление различает стили.

In [ ]:
proj = {}
for name in ['CLAP-эмбеддинг', 'Семантические теги', 'Гибрид']:
    X = REPS[name]
    proj[(name, 'UMAP')] = umap.UMAP(n_neighbors=10, min_dist=0.3, metric='cosine', random_state=SEED).fit_transform(X)
    proj[(name, 't-SNE')] = TSNE(n_components=2, perplexity=min(15, N // 4), metric='cosine', init='pca',
                                 random_state=SEED).fit_transform(X)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for j, name in enumerate(['CLAP-эмбеддинг', 'Семантические теги', 'Гибрид']):
    for i, method in enumerate(['UMAP', 't-SNE']):
        scatter_by_genre(axes[i, j], proj[(name, method)], genres, f'{method}: {name}')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=12, fontsize=9, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=(0, 0.03, 1, 1)); savefig('11_embedding_space'); plt.show()

In [ ]:
xy = proj[('Гибрид', 'UMAP')]
fig, ax = plt.subplots(figsize=(15, 11))
scatter_by_genre(ax, xy, genres, 'UMAP гибридного представления (подписаны исполнители)')
for (x, y), a in zip(xy, features['artist']):
    ax.annotate(a[:18], (x, y), fontsize=6.5, color='#52514e', xytext=(3, 3), textcoords='offset points')
ax.legend(loc='best', fontsize=8, ncol=2)
plt.tight_layout(); savefig('12_umap_labeled'); plt.show()

# тот же UMAP, но цвет по настроению
fig, ax = plt.subplots(figsize=(9, 7))
moods_top = features['mood_top'].to_numpy(dtype=str)
for i, m in enumerate(sorted(set(moods_top))):
    sel = moods_top == m
    ax.scatter(xy[sel, 0], xy[sel, 1], s=46, c=CAT[i % 8], marker='o' if i < 8 else '^', label=m,
               edgecolors='#fcfcfb', linewidths=1.2)
ax.set_title('UMAP гибрида: цвет — настроение (CLAP)'); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
ax.legend(fontsize=8)
plt.tight_layout(); savefig('13_umap_by_mood'); plt.show()

### 5.2 Ближайшие треки
5 соседей в гибридном пространстве для нескольких треков. По ссылкам можно послушать и проверить.

In [ ]:
def show_neighbors(track_id, rep='Гибрид', k=5):
    X = REPS[rep]
    i = features.index[features.track_id == track_id][0]
    idx, sims = neighbors(X, k)
    q = features.loc[i]
    print(f"▶ {q.artist} — {q.title}  [{GENRE_RU[q.genre]}, {q.year}, настроение: {q.mood_top}, контекст: {q.ctx_top}]")
    out = features.loc[idx[i], ['artist', 'title', 'genre', 'year', 'mood_top', 'ctx_top', 'clip_url']].copy()
    out.insert(0, 'сходство', sims[i].round(3))
    out['genre'] = out['genre'].map(GENRE_RU)
    display(out.reset_index(drop=True))

query_ids = ['rock_03', 'electronic_01', 'jazz_02', 'classical_05', 'hip_hop_02', 'ambient_05', 'country_02', 'metal_01']
for t in query_ids:
    if t in set(features.track_id):
        show_neighbors(t)

In [ ]:
# один и тот же трек в разных представлениях
t = 'jazz_02' if 'jazz_02' in set(features.track_id) else features.track_id.iloc[0]
for rep in ['CLAP-эмбеддинг', 'Семантические теги', 'Сигнальные признаки']:
    print(f'\n=== {rep} ===')
    show_neighbors(t, rep=rep, k=5)

### 5.3 Похожие артисты
У каждого исполнителя один трек, так что похожие артисты здесь просто авторы трёх ближайших треков.

In [ ]:
idx, sims = neighbors(REPS['Гибрид'], 3)
features['similar_artists_emb'] = ['|'.join(features.loc[row, 'artist']) for row in idx]
sim_art = features[['artist', 'genre', 'similar_artists_emb']].copy()
if 'llm_similar_artists' in features.columns:
    sim_art['llm_similar_artists'] = features['llm_similar_artists']
sim_art.to_csv(DATA_DIR / 'similar_artists.csv', index=False)
sim_art['genre'] = sim_art['genre'].map(GENRE_RU)
sim_art.sample(12, random_state=SEED)

### 5.4 Какие признаки полезнее
1. Убираю из гибрида по одному блоку и смотрю, насколько падает Genre P@5.
2. Считаю kNN отдельно по каждой группе признаков.
3. Обучаю Random Forest предсказывать жанр (кросс-валидация на 4 фолда) и смотрю важность признаков. Для сравнения логистическая регрессия на эмбеддинге.

In [ ]:
abl = [{'конфигурация': 'полный гибрид', 'Genre P@5': precision_at_k(hybrid(), genres), 'Era P@5': precision_at_k(hybrid(), eras)}]
for k, name in (('clap', 'без CLAP-эмбеддинга'), ('sem', 'без семантических тегов'), ('sig', 'без сигнальных признаков')):
    w = {**W, k: 0}
    abl.append({'конфигурация': name, 'Genre P@5': precision_at_k(hybrid(w), genres), 'Era P@5': precision_at_k(hybrid(w), eras)})
abl = pd.DataFrame(abl).set_index('конфигурация')
abl['Δ Genre P@5'] = abl['Genre P@5'] - abl.loc['полный гибрид', 'Genre P@5']
print(abl.round(3))

groups = {'настроение': MOODS, 'контекст': CTXS, 'атмосфера': ATMS, 'инструменты + вокал': INSTS,
          'темп + энергия': ['tempo_bpm', 'rms_mean', 'onset_rate', 'pulse_clarity'],
          'тембр (яркость, шумность, ZCR)': ['spectral_centroid_hz', 'spectral_flatness', 'zcr_mean'],
          'популярность': ['log_views']}
grp = pd.DataFrame([{'группа признаков': g, 'число признаков': len(cols),
                     'Genre P@5': precision_at_k(l2n(zs(features[cols].values)), genres),
                     'Era P@5': precision_at_k(l2n(zs(features[cols].values)), eras)} for g, cols in groups.items()])
grp = grp.sort_values('Genre P@5', ascending=False).set_index('группа признаков')
grp.round(3)

In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
X_int = np.nan_to_num(features[SEMANTIC + SIGNAL].values.astype(float))
rf = RandomForestClassifier(n_estimators=500, random_state=SEED, min_samples_leaf=2)
acc_sets = {
    'RF: семантика + сигнал': cross_val_score(rf, X_int, genres, cv=cv).mean(),
    'RF: только семантика': cross_val_score(rf, np.nan_to_num(features[SEMANTIC].values), genres, cv=cv).mean(),
    'RF: только сигнал': cross_val_score(rf, np.nan_to_num(features[SIGNAL].values), genres, cv=cv).mean(),
    'LogReg: CLAP-эмбеддинг': cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5)),
                                              E, genres, cv=cv).mean(),
}
print(pd.Series(acc_sets, name='accuracy').round(3))

rf.fit(X_int, genres)
imp = pd.Series(rf.feature_importances_, index=SEMANTIC + SIGNAL).sort_values(ascending=True).tail(15)
fig, ax = plt.subplots(figsize=(9, 5.5))
colors = [CAT[1] if c in SIGNAL else ACCENT for c in imp.index]
ax.barh(imp.index, imp.values, color=colors, height=0.6)
ax.set_title('Важность признаков (Random Forest)'); ax.set_xlabel('importance')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=ACCENT, label='CLAP'), Patch(color=CAT[1], label='librosa')],
          loc='lower right', fontsize=8)
plt.tight_layout(); savefig('14_feature_importance'); plt.show()

### 5.5 Итог по экспериментам

In [ ]:
best_rep = results.drop('Случайный')['Genre P@5'].idxmax()
most_important_block = abl['Δ Genre P@5'].drop('полный гибрид').idxmin()
print(f"лучшее представление: {best_rep} ({results.loc[best_rep, 'Genre P@5']:.2f})")
print(f"сильнее всего падает: {most_important_block} ({abl.loc[most_important_block, 'Δ Genre P@5']:+.2f})")
print(f"лучшая группа по жанру: {grp.index[0]} ({grp.iloc[0]['Genre P@5']:.2f})")
print(f"лучшая группа по эпохе: {grp['Era P@5'].idxmax()} ({grp['Era P@5'].max():.2f})")
print('топ-5 признаков:', ', '.join(imp.sort_values(ascending=False).index[:5]))

exp = {'representations': results.round(4).to_dict(), 'ablation': abl.round(4).to_dict(),
       'feature_groups': grp.round(4).to_dict(), 'genre_cv_accuracy': {k: round(v, 4) for k, v in acc_sets.items()},
       'top_features': imp.sort_values(ascending=False).round(4).to_dict(), 'hybrid_weights': W}
(DATA_DIR / 'experiment_results.json').write_text(json.dumps(exp, ensure_ascii=False, indent=2), encoding='utf-8')
features.to_csv(DATA_DIR / 'features.csv', index=False)
